# VAJRA — Kaggle T4×2 llama.cpp Worker

Uses **llama.cpp + Qwen2.5-Coder-32B-Instruct Q4_K_M** instead of Ollama. The model is served through llama.cpp's OpenAI-compatible API and wrapped by a VAJRA worker endpoint, so the existing VAJRA control plane can use it without changing authority semantics.

Use a Kaggle **T4×2 GPU** runtime with Internet enabled. The notebook can reuse an attached model dataset if it contains a Q4_K_M GGUF; otherwise it downloads the official Qwen Q4_K_M file from Hugging Face. The supplied FP16 Kaggle dataset is intentionally not selected: FP16 32B is about 64 GB and does not fit in 2×T4 VRAM. The supplied BNB 4-bit dataset is a Transformers/bitsandbytes path, not the llama.cpp GGUF path.

In [ ]:
%%bash
set -euo pipefail

REPO=/kaggle/working/vajra
MODEL_DIR=/kaggle/working/models
MODEL_FILE="$MODEL_DIR/qwen2.5-coder-32b-instruct-q4_k_m.gguf"
LLAMA_DIR=/kaggle/working/llama.cpp
LLAMA_TAG=b10982

echo '=== VAJRA LLAMA.CPP T4x2 WORKER ==='
if [ ! -d "$REPO/.git" ]; then git clone -q https://github.com/Exploiter69/vajra.git "$REPO"; else git -C "$REPO" fetch -q origin main && git -C "$REPO" reset --hard -q origin/main; fi
cd "$REPO"; export PYTHONPATH="$REPO/src"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

mkdir -p "$MODEL_DIR"
ATTACHED=$(find /kaggle/input -type f \( -iname '*q4_k_m*.gguf' -o -iname '*Q4_K_M*.gguf' \) 2>/dev/null | head -1 || true)
if [ -n "$ATTACHED" ]; then
  echo "Using attached Q4_K_M: $ATTACHED"; MODEL_FILE="$ATTACHED"
elif [ ! -f "$MODEL_FILE" ]; then
  echo 'Downloading official Qwen Q4_K_M GGUF (~20 GB)...'
  python -m pip -q install -U huggingface_hub
  python - <<'PY'
from huggingface_hub import hf_hub_download
p = hf_hub_download(repo_id='Qwen/Qwen2.5-Coder-32B-Instruct-GGUF', filename='qwen2.5-coder-32b-instruct-q4_k_m.gguf', local_dir='/kaggle/working/models')
print(p)
PY
fi

if [ ! -x "$LLAMA_DIR/llama-server" ]; then
  echo "Downloading llama.cpp CUDA 13 binary $LLAMA_TAG..."
  mkdir -p "$LLAMA_DIR"
  URL="https://github.com/ggml-org/llama.cpp/releases/download/$LLAMA_TAG/llama-$LLAMA_TAG-bin-ubuntu-cuda-13-x64.tar.gz"
  curl -fL "$URL" -o /tmp/llama.tar.gz
  tar -xzf /tmp/llama.tar.gz -C "$LLAMA_DIR" --strip-components=1
  find "$LLAMA_DIR" -maxdepth 3 -type f -name 'llama-server' -exec cp {} "$LLAMA_DIR/llama-server" \;
  chmod +x "$LLAMA_DIR/llama-server"
fi
echo '=== LLAMA.CPP DEVICES ==='
"$LLAMA_DIR/llama-server" --list-devices || true
echo '=== STARTING MODEL ==='
export VAJRA_WORKER_HOST=127.0.0.1 VAJRA_WORKER_PORT=8787 VAJRA_LLAMA_URL=http://127.0.0.1:8000 VAJRA_WORKER_MODEL=qwen2.5-coder-32b VAJRA_WORKER_ID=kaggle-t4-llama-cpp-01
"$LLAMA_DIR/llama-server" -m "$MODEL_FILE" --host 127.0.0.1 --port 8000 --n-gpu-layers all --split-mode layer --ctx-size 8192 --parallel 1 > /kaggle/working/llama-server.log 2>&1 &
LLAMA_PID=$!
for i in $(seq 1 180); do curl -fsS --max-time 3 http://127.0.0.1:8000/health >/dev/null 2>&1 && break; kill -0 "$LLAMA_PID" 2>/dev/null || { cat /kaggle/working/llama-server.log; exit 1; }; sleep 1; done
curl -fsS http://127.0.0.1:8000/health

echo '=== START VAJRA WORKER ==='
python -m vajra.runtime.llama_cpp_worker_server > /kaggle/working/vajra-worker.log 2>&1 &
WORKER_PID=$!
for i in $(seq 1 60); do curl -fsS --max-time 3 http://127.0.0.1:8787/health >/tmp/vajra-health.json 2>/dev/null && break; kill -0 "$WORKER_PID" 2>/dev/null || { cat /kaggle/working/vajra-worker.log; exit 1; }; sleep 1; done
cat /tmp/vajra-health.json

echo '=== REAL VAJRA WORKER INFERENCE ==='
python - <<'PY'
import json, subprocess, urllib.request, uuid
from vajra.runtime.worker_protocol import WorkerJob
from vajra.runtime.kaggle_worker import KaggleWorkerAdapter
job=WorkerJob(run_id='llama-smoke',step_id='smoke',attempt_id='1',repository_revision=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip(),workspace_contract={'mode':'read_only_smoke'},context_bundle={'prompt':'Return exactly VAJRA_LLAMA_CPP_OK'},allowed_capabilities=('completion',),budget={'max_output_tokens':32,'timeout_seconds':180},deadline='2099-01-01T00:00:00Z',expected_output_schema={'type':'string'},correlation_id=str(uuid.uuid4()))
req=urllib.request.Request('http://127.0.0.1:8787/infer',data=KaggleWorkerAdapter().encode_job(job).encode(),headers={'Content-Type':'application/json'},method='POST')
with urllib.request.urlopen(req,timeout=190) as r: print(r.read().decode())
PY
echo '=== VAJRA LLAMA.CPP WORKER READY ==='
echo 'Keep this cell running while VAJRA uses the worker.'
wait "$LLAMA_PID"
